# Camada Gold — V-Commerce CRM 360

**Módulo 1 · Engenharia de Dados · Arquitetura Medalhão**

---

## Visão Geral

A camada Gold consolida e agrega os dados tratados da Silver para responder perguntas de negócio específicas. As tabelas produzidas aqui são **desnormalizadas por design** — o objetivo é que o backend FastAPI execute o mínimo de JOINs possível, já que cada tabela chega pronta para consumo pelos endpoints do CRM.

Os princípios aplicados em todas as tabelas Gold são:

- **Agregação orientada ao consumidor** — cada tabela responde a uma pergunta de negócio clara, mapeada a um stakeholder específico
- **Idempotência** — todas as escritas usam `mode=overwrite`, garantindo que reexecutar produza sempre o mesmo resultado
- **Rastreabilidade** — `timestamp_ingestion` registra o instante em que a tabela Gold foi gerada
- **Sem JOINs no backend** — métricas calculáveis a partir da Silver são materializadas aqui para evitar recomputação em tempo de consulta
- **Leitura exclusiva da Silver** — nenhuma tabela Gold lê de Bronze diretamente

---

## Tabelas produzidas neste notebook

| Tabela Gold | Granularidade | Stakeholder Principal | Descrição |
|---|---|---|---|
| `gold_cliente_360` | 1 linha por cliente | Fernanda Souza (Customer Success) | Visão 360 consolidada de cada cliente |
| `gold_kpis_vendas_mensal` | 1 linha por mês | Ricardo Alves (Diretor Comercial) | KPIs de vendas mensais para o dashboard |
| `gold_vendas_por_dimensao` | 1 linha por mês x região x categoria | Ricardo Alves (Diretor Comercial) | Drill-down de receita por dimensão |
| `gold_desempenho_produto` | 1 linha por produto | Marcelo Teixeira (Gerente de Produto) | Métricas individuais de cada produto |
| `gold_analise_suporte_por_tipo` | 1 linha por tipo de problema | Time de Suporte | Desempenho do SAC por categoria de problema |
| `gold_analise_suporte_por_agente` | 1 linha por agente | Gestão do SAC | Desempenho individual de cada agente |
| `gold_satisfacao_nps` | 1 linha por mês x categoria | Diretoria | NPS e satisfação consolidados por período e categoria |
| `gold_pedidos_detalhado` | 1 linha por pedido/produto | Operações / CRM / Agente de IA | Base detalhada de pedidos, enriquecida com produto e valor final padronizado |
| `gold_pedidos_financeiro` | 1 linha por pedido/produto calculável | Diretoria Comercial / Financeiro | Base financeira apenas com pedidos com quantidade e preço válidos |
| `gold_pedidos_por_status` | 1 linha por mês x status | Operações / Diretoria Comercial | Distribuição operacional dos pedidos por status ao longo do tempo |
| `gold_vendas_mensais` | 1 linha por mês x método de pagamento | Diretoria Comercial / Gestão Executiva | Métricas mensais de vendas, receita, reembolsos e ticket médio |
| `gold_pedidos_cliente` | 1 linha por cliente | Customer Success / CRM / Agente de IA | Histórico operacional de pedidos por cliente, sem métricas financeiras |

---

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

catalogo      = 'vcommerce_catalog'
silver_schema = 'vcommerce_silver'
gold_schema   = 'vcommerce_gold'

spark.sql(f'USE CATALOG {catalogo}')
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {gold_schema}')
spark.sql(f'USE SCHEMA {gold_schema}')

print(f'Catálogo : {catalogo}')
print(f'Silver   : {silver_schema}')
print(f'Gold     : {gold_schema}')

Catálogo : vcommerce_catalog
Silver   : vcommerce_silver
Gold     : vcommerce_gold


---

## Tabela: `gold_cliente_360`

**Origem:** `silver.dim_clientes`, `silver.ft_pedidos`, `silver.ft_tickets_suporte`, `silver.ft_avaliacoes`  
**Destino:** `gold.gold_cliente_360`  
**Stakeholder:** Fernanda Souza — Diretora de Customer Success  
**Descrição:** Visão consolidada de cada cliente em uma única linha, agregando métricas de compra, suporte e satisfação. É a tabela que alimenta o perfil 360 do cliente no CRM.

### Schema

| Coluna | Tipo | Origem | Descrição |
|---|---|---|---|
| `id_cliente` | string | dim_clientes | Chave primária |
| `nome_completo` | string | dim_clientes | Nome completo do cliente |
| `email` | string | dim_clientes | E-mail de contato |
| `regiao` | string | dim_clientes | Região geográfica |
| `origem` | string | dim_clientes | Canal de aquisição (Web, Indicação…) |
| `total_pedidos` | long | ft_pedidos | Total de pedidos realizados |
| `receita_total` | decimal | ft_pedidos | Soma de `valor_pedido` de todos os pedidos |
| `ticket_medio` | decimal | ft_pedidos | `receita_total / total_pedidos` |
| `data_primeiro_pedido` | date | ft_pedidos | Data do primeiro pedido |
| `data_ultimo_pedido` | date | ft_pedidos | Data do pedido mais recente |
| `metodo_pagamento_favorito` | string | ft_pedidos | Método de pagamento mais utilizado (moda) |
| `total_tickets` | long | ft_tickets_suporte | Total de tickets abertos |
| `taxa_resolucao` | decimal | ft_tickets_suporte | `tickets_resolvidos / total_tickets` |
| `nota_media_atendimento` | decimal | ft_tickets_suporte | Média de `nota_avaliacao` dos tickets |
| `nota_nps_media` | decimal | ft_avaliacoes | Média das notas NPS dadas pelo cliente |
| `categoria_nps_predominante` | string | ft_avaliacoes | Categoria NPS mais frequente (Promotor/Neutro/Detrator) |
| `nota_produto_media` | decimal | ft_avaliacoes | Média das notas de produto dadas pelo cliente |
| `segmento_cliente` | string | derivado | VIP / Ativo / Em risco / Inativo |
| `timestamp_ingestion` | timestamp | derivado | Instante de geração desta tabela Gold |

### Regras de segmentação (`segmento_cliente`)

| Segmento   | Critério |
---|---|
 `VIP`      | `receita_total >= 2000` ou `total_pedidos >= 10` |
 `Ativo`    | Último pedido há no máximo 90 dias (e não VIP) |
 `Em risco` | Último pedido entre 90 e 180 dias atrás |
 `Inativo`  | Último pedido há mais de 180 dias |
 `Lead`     | Cliente sem nenhum pedido |

In [0]:
# ─── Leitura das tabelas Silver ───────────────────────────────────────────────
clientes  = spark.table(f'{silver_schema}.dim_clientes')
pedidos   = spark.table(f'{silver_schema}.ft_pedidos')
tickets   = spark.table(f'{silver_schema}.ft_tickets_suporte')
avaliacoes = spark.table(f'{silver_schema}.ft_avaliacoes')

# ─── Agregação de pedidos por cliente ─────────────────────────────────────────
# metodo_pagamento_favorito: modo (valor mais frequente) por cliente
w_pagto = Window.partitionBy('id_cliente').orderBy(F.desc('freq_pagto'))

metodo_fav = (
    pedidos
    .groupBy('id_cliente', 'metodo_pagamento')
    .agg(F.count('*').alias('freq_pagto'))
    .withColumn('rank_pagto', F.row_number().over(w_pagto))
    .filter(F.col('rank_pagto') == 1)
    .select('id_cliente', F.col('metodo_pagamento').alias('metodo_pagamento_favorito'))
)

# contagem de pedidos do cliente e contagem de produtos distintos
agg_pedidos = (
    pedidos
    .groupBy('id_cliente')
    .agg(
        F.count('id_pedido').alias('total_pedidos'),
        F.countDistinct('id_produto').alias('total_produtos_distintos'),
        F.round(F.sum('valor_pedido'), 2).alias('receita_total'),
        F.round(F.avg('valor_pedido'), 2).alias('ticket_medio'),
        F.min(F.to_date('data_pedido')).alias('data_primeiro_pedido'),
        F.max(F.to_date('data_pedido')).alias('data_ultimo_pedido'),
    )
    .join(metodo_fav, on='id_cliente', how='left')
)

# ─── Agregação de tickets por cliente ─────────────────────────────────────────
agg_tickets = (
    tickets
    .groupBy('id_cliente')
    .agg(
        F.count('ticket_id').alias('total_tickets'),
        F.round(
            F.sum(F.when(F.col('resolvido') == True, 1).otherwise(0)) / F.count('ticket_id'),
            4
        ).alias('taxa_resolucao'),
        F.round(F.avg('nota_avaliacao'), 2).alias('nota_media_atendimento'),
    )
)

# ─── Agregação de avaliações por cliente ──────────────────────────────────────
# categoria_nps_predominante: modo por cliente
w_nps = Window.partitionBy('id_cliente').orderBy(F.desc('freq_nps'))

nps_fav = (
    avaliacoes
    .groupBy('id_cliente', 'categoria_nps')
    .agg(F.count('*').alias('freq_nps'))
    .withColumn('rank_nps', F.row_number().over(w_nps))
    .filter(F.col('rank_nps') == 1)
    .select(
        'id_cliente',
        F.col('categoria_nps').alias('categoria_nps_predominante')
    )
)

# Calcula a categoria NPS da última avaliação de cada cliente,
# retornando uma coluna 'categoria_nps_recente' por 'id_cliente'.
nps_recente = (
    avaliacoes
    .withColumn('data_avaliacao', F.to_timestamp('data_avaliacao'))
    .withColumn('rank_nps', F.row_number().over(Window.partitionBy('id_cliente').orderBy(F.desc('data_avaliacao'))))
    .filter(F.col('rank_nps') == 1)
    .select('id_cliente', F.col('categoria_nps').alias('categoria_nps_recente'))
)

agg_avaliacoes = (
    avaliacoes
    .groupBy('id_cliente')
    .agg(
        F.round(F.avg('nota_nps'), 2).alias('nota_nps_media'),
        F.round(F.avg('nota_produto'), 2).alias('nota_produto_media'),
    )
    .join(nps_fav, on='id_cliente', how='left')
)

# ─── Base de clientes + joins das agregações ──────────────────────────────────
base = clientes.select(
    'id_cliente', 'nome_completo', 'email', 'regiao', 'origem'
)

gold_c360 = (
    base
    .join(agg_pedidos,   on='id_cliente', how='left')
    .join(agg_tickets,   on='id_cliente', how='left')
    .join(agg_avaliacoes, on='id_cliente', how='left')
)

# ─── Derivação: segmento_cliente ──────────────────────────────────────────────
dias_inativo = F.datediff(F.current_date(), F.col('data_ultimo_pedido'))

gold_c360 = gold_c360.withColumn(
    'segmento_cliente',
    F.when(
        (F.col('receita_total') >= 2000) | (F.col('total_pedidos') >= 10), 'VIP'
    ).when(
        dias_inativo <= 90, 'Ativo'
    ).when(
        (dias_inativo > 90) & (dias_inativo <= 180), 'Em risco'
    ).otherwise('Inativo')
)

# ─── Timestamp de geração ─────────────────────────────────────────────────────
gold_c360 = gold_c360.withColumn('timestamp_ingestion', F.current_timestamp())

# ─── Escrita ──────────────────────────────────────────────────────────────────
gold_c360.write.format('delta').mode('overwrite').saveAsTable(f'{gold_schema}.gold_cliente_360')

total = gold_c360.count()
print(f'gold_cliente_360 gravada: {total:,} clientes')
print('Distribuição por segmento:')
gold_c360.groupBy('segmento_cliente').count().orderBy(F.desc('count')).show()

gold_cliente_360 gravada: 58,322 clientes
Distribuição por segmento:
+----------------+-----+
|segmento_cliente|count|
+----------------+-----+
|             VIP|43463|
|         Inativo|12337|
|           Ativo| 1776|
|        Em risco|  746|
+----------------+-----+



---

## Tabela: `gold_kpis_vendas_mensal`

**Origem:** `silver.ft_pedidos`, `silver.dim_clientes`  
**Destino:** `gold.gold_kpis_vendas_mensal`  
**Stakeholder:** Ricardo Alves — Diretor Comercial  
**Descrição:** Uma linha por mês com os principais KPIs de vendas. Alimenta o gráfico de série temporal e o painel de KPIs do dashboard.

### Schema

| Coluna | Tipo | Origem | Descrição |
|---|---|---|---|
| `ano_mes` | string | ft_pedidos | Período no formato `YYYY-MM` |
| `receita_total` | decimal | ft_pedidos | Soma de `valor_pedido` dos pedidos não cancelados |
| `total_pedidos` | long | ft_pedidos | Total de pedidos no período |
| `ticket_medio` | decimal | ft_pedidos | `receita_total / total_pedidos` |
| `total_clientes_ativos` | long | ft_pedidos | Clientes distintos com pedido no período |
| `novos_clientes` | long | ft_pedidos | Clientes cujo primeiro pedido ocorreu neste mês |
| `pedidos_cancelados` | long | ft_pedidos | Pedidos com status cancelado |
| `taxa_cancelamento` | decimal | ft_pedidos | `pedidos_cancelados / total_pedidos` |
| `timestamp_ingestion` | timestamp | derivado | Instante de geração desta tabela Gold |

In [0]:
pedidos = spark.table(f'{silver_schema}.ft_pedidos')

# ─── Primeiro pedido por cliente (para calcular novos_clientes) ───────────────
primeiro_pedido = (
    pedidos
    .groupBy('id_cliente')
    .agg(F.min('ano_mes').alias('ano_mes_primeiro_pedido'))
)

# ─── Novos clientes por mês ───────────────────────────────────────────────────
novos_por_mes = (
    primeiro_pedido
    .groupBy('ano_mes_primeiro_pedido')
    .agg(F.count('id_cliente').alias('novos_clientes'))
    .withColumnRenamed('ano_mes_primeiro_pedido', 'ano_mes')
)

# ─── KPIs mensais ─────────────────────────────────────────────────────────────
pedidos_nao_cancelados = pedidos.filter(F.col('status') != 'Cancelado')

kpis = (
    pedidos
    .groupBy('ano_mes')
    .agg(
        F.round(F.sum(
            F.when(F.col('status') != 'Cancelado', F.col('valor_pedido')).otherwise(0)
        ), 2).alias('receita_total'),
        F.count('id_pedido').alias('total_pedidos'),
        F.countDistinct('id_cliente').alias('total_clientes_ativos'),
        F.sum(
            F.when(F.col('status') == 'Cancelado', 1).otherwise(0)
        ).alias('pedidos_cancelados'),
    )
    .join(novos_por_mes, on='ano_mes', how='left')
    .withColumn('novos_clientes', F.coalesce(F.col('novos_clientes'), F.lit(0)))
    .withColumn(
        'ticket_medio',
        F.round(F.col('receita_total') / F.col('total_pedidos'), 2)
    )
    .withColumn(
        'taxa_cancelamento',
        F.round(F.col('pedidos_cancelados') / F.col('total_pedidos'), 4)
    )
    .withColumn('timestamp_ingestion', F.current_timestamp())
    .orderBy('ano_mes')
)

kpis.write.format('delta').mode('overwrite').saveAsTable(f'{gold_schema}.gold_kpis_vendas_mensal')

print(f'gold_kpis_vendas_mensal gravada: {kpis.count()} meses')
kpis.select('ano_mes', 'receita_total', 'total_pedidos', 'ticket_medio', 'novos_clientes', 'taxa_cancelamento').show(5)

gold_kpis_vendas_mensal gravada: 41 meses
+-------+-------------+-------------+------------+--------------+-----------------+
|ano_mes|receita_total|total_pedidos|ticket_medio|novos_clientes|taxa_cancelamento|
+-------+-------------+-------------+------------+--------------+-----------------+
|2023-01|   9974847.96|         5420|     1840.38|          5042|              0.0|
|2023-02|   8317228.73|         4829|     1722.35|          4063|              0.0|
|2023-03|  11291671.69|         6195|     1822.71|          4621|              0.0|
|2023-04|  11318951.18|         6343|     1784.48|          4118|              0.0|
|2023-05|  12581658.28|         6709|     1875.34|          3880|              0.0|
+-------+-------------+-------------+------------+--------------+-----------------+
only showing top 5 rows


---

## Tabela: `gold_vendas_por_dimensao`

**Origem:** `silver.ft_pedidos`, `silver.dim_produtos`, `silver.dim_clientes`  
**Destino:** `gold.gold_vendas_por_dimensao`  
**Stakeholder:** Ricardo Alves — Diretor Comercial  
**Descrição:** Drill-down de vendas com granularidade `ano_mes x regiao x categoria`. Permite ao Diretor Comercial investigar qual região ou categoria está impactando os resultados de um determinado período.

### Schema

| Coluna | Tipo | Origem | Descrição |
|---|---|---|---|
| `ano_mes` | string | ft_pedidos | Período no formato `YYYY-MM` |
| `regiao` | string | dim_clientes | Região geográfica do cliente |
| `categoria` | string | dim_produtos | Categoria do produto vendido |
| `receita_total` | decimal | ft_pedidos | Soma de `valor_pedido` (pedidos não cancelados) |
| `total_pedidos` | long | ft_pedidos | Total de pedidos na combinação |
| `ticket_medio` | decimal | derivado | `receita_total / total_pedidos` |
| `quantidade_itens_vendidos` | long | ft_pedidos | Soma de `quantidade` |
| `timestamp_ingestion` | timestamp | derivado | Instante de geração desta tabela Gold |

In [0]:
pedidos   = spark.table(f'{silver_schema}.ft_pedidos')
produtos  = spark.table(f'{silver_schema}.dim_produtos').select('id_produto', 'categoria')
clientes  = spark.table(f'{silver_schema}.dim_clientes').select('id_cliente', 'regiao')

dim = (
    pedidos
    .filter(F.col('status') != 'Cancelado')
    .join(produtos, on='id_produto', how='left')
    .join(clientes, on='id_cliente', how='left')
    .groupBy('ano_mes', 'regiao', 'categoria')
    .agg(
        F.round(F.sum('valor_pedido'), 2).alias('receita_total'),
        F.count('id_pedido').alias('total_pedidos'),
        F.sum('quantidade').alias('quantidade_itens_vendidos'),
    )
    .withColumn(
        'ticket_medio',
        F.round(F.col('receita_total') / F.col('total_pedidos'), 2)
    )
    .withColumn('timestamp_ingestion', F.current_timestamp())
    .orderBy('ano_mes', 'regiao', 'categoria')
)

dim.write.format('delta').mode('overwrite').saveAsTable(f'{gold_schema}.gold_vendas_por_dimensao')

print(f'gold_vendas_por_dimensao gravada: {dim.count():,} combinações')
dim.orderBy(F.desc('receita_total')).show(5)

gold_vendas_por_dimensao gravada: 2,213 combinações
+-------+--------+-----------+-------------+-------------+-------------------------+------------+--------------------+
|ano_mes|  regiao|  categoria|receita_total|total_pedidos|quantidade_itens_vendidos|ticket_medio| timestamp_ingestion|
+-------+--------+-----------+-------------+-------------+-------------------------+------------+--------------------+
|2024-11| Sudeste|Eletronicos|  10826771.04|         2867|                     7948|     3776.34|2026-05-06 06:33:...|
|2024-12| Sudeste|Eletronicos|   8950150.29|         2272|                     6307|     3939.33|2026-05-06 06:33:...|
|2024-06| Sudeste|Eletronicos|   8868185.55|         2355|                     6564|     3765.68|2026-05-06 06:33:...|
|2024-11|Nordeste|Eletronicos|   7312228.55|         1834|                     5250|     3987.04|2026-05-06 06:33:...|
|2024-10| Sudeste|Eletronicos|   6347320.40|         1699|                     4783|     3735.92|2026-05-06 06:33:.

---

## Tabela: `gold_desempenho_produto`

**Origem:** `silver.dim_produtos`, `silver.ft_pedidos`, `silver.ft_avaliacoes`, `silver.ft_tickets_suporte`  
**Destino:** `gold.gold_desempenho_produto`  
**Stakeholder:** Marcelo Teixeira — Gerente de Produto  
**Descrição:** Uma linha por produto com métricas consolidadas de vendas, satisfação e suporte. O campo `ratio_ticket_por_venda` responde à dor central do Gerente de Produto: identificar produtos que vendem bem mas geram custo operacional desproporcional via tickets de suporte.

### Schema

| Coluna | Tipo | Origem | Descrição |
|---|---|---|---|
| `id_produto` | string | dim_produtos | Chave primária |
| `nome_produto` | string | dim_produtos | Nome do produto |
| `categoria` | string | dim_produtos | Categoria normalizada |
| `preco` | decimal | dim_produtos | Preço de tabela atual |
| `fornecedor` | string | dim_produtos | Fornecedor do produto |
| `estoque_disponivel` | int | dim_produtos | Unidades em estoque |
| `ativo` | boolean | dim_produtos | Se o produto está ativo no catálogo |
| `receita_total` | decimal | ft_pedidos | Receita gerada pelo produto |
| `qtd_vendida` | long | ft_pedidos | Total de unidades vendidas |
| `ticket_medio` | decimal | ft_pedidos | Receita média por pedido |
| `nota_media_avaliacao` | decimal | ft_avaliacoes | Média das notas de produto |
| `qtd_avaliacoes` | long | ft_avaliacoes | Total de avaliações recebidas |
| `nota_nps_media` | decimal | ft_avaliacoes | Média das notas NPS vinculadas ao produto |
| `qtd_tickets_gerados` | long | ft_tickets_suporte | Tickets de suporte originados de pedidos deste produto |
| `tipo_problema_mais_frequente` | string | ft_tickets_suporte | Problema mais comum relatado para este produto |
| `ratio_ticket_por_venda` | decimal | derivado | `qtd_tickets_gerados / qtd_vendida` — índice de problema por unidade vendida |
| `timestamp_ingestion` | timestamp | derivado | Instante de geração desta tabela Gold |

In [0]:
produtos   = spark.table(f'{silver_schema}.dim_produtos')
pedidos    = spark.table(f'{silver_schema}.ft_pedidos')
avaliacoes = spark.table(f'{silver_schema}.ft_avaliacoes')
tickets    = spark.table(f'{silver_schema}.ft_tickets_suporte')

# ─── Métricas de vendas por produto ───────────────────────────────────────────
agg_vendas = (
    pedidos
    .filter(F.col('status') != 'Cancelado')
    .groupBy('id_produto')
    .agg(
        F.round(F.sum('valor_pedido'), 2).alias('receita_total'),
        F.sum('quantidade').alias('qtd_vendida'),
        F.round(F.avg('valor_pedido'), 2).alias('ticket_medio'),
    )
)

# ─── Métricas de avaliações por produto ───────────────────────────────────────
agg_aval = (
    avaliacoes
    .groupBy('id_produto')
    .agg(
        F.round(F.avg('nota_produto'), 2).alias('nota_media_avaliacao'),
        F.count('id_avaliacao').alias('qtd_avaliacoes'),
        F.round(F.avg('nota_nps'), 2).alias('nota_nps_media'),
    )
)

# ─── Tickets via pedidos - produto ────────────────────────────────────────────
# Tickets não têm id_produto diretamente: linkamos via id_pedido - ft_pedidos
tickets_com_produto = (
    tickets
    .join(
        pedidos.select('id_pedido', 'id_produto'),
        on='id_pedido', how='left'
    )
)

# Tipo de problema mais frequente por produto
w_prob = Window.partitionBy('id_produto').orderBy(F.desc('freq_prob'))

tipo_fav = (
    tickets_com_produto
    .groupBy('id_produto', 'tipo_problema')
    .agg(F.count('*').alias('freq_prob'))
    .withColumn('rank_prob', F.row_number().over(w_prob))
    .filter(F.col('rank_prob') == 1)
    .select('id_produto', F.col('tipo_problema').alias('tipo_problema_mais_frequente'))
)

agg_tickets = (
    tickets_com_produto
    .groupBy('id_produto')
    .agg(F.count('ticket_id').alias('qtd_tickets_gerados'))
    .join(tipo_fav, on='id_produto', how='left')
)

# ─── Join final ───────────────────────────────────────────────────────────────
gold_prod = (
    produtos.select(
        'id_produto', 'nome_produto', 'categoria', 'preco',
        'fornecedor', 'estoque_disponivel', 'ativo'
    )
    .join(agg_vendas,   on='id_produto', how='left')
    .join(agg_aval,     on='id_produto', how='left')
    .join(agg_tickets,  on='id_produto', how='left')
    .withColumn(
        'ratio_ticket_por_venda',
        F.round(
            F.col('qtd_tickets_gerados') / F.col('qtd_vendida'),
            4
        )
    )
    .withColumn('timestamp_ingestion', F.current_timestamp())
)

gold_prod.write.format('delta').mode('overwrite').saveAsTable(f'{gold_schema}.gold_desempenho_produto')

print(f'gold_desempenho_produto gravada: {gold_prod.count():,} produtos')
print('Top 5 por ratio_ticket_por_venda:')
gold_prod.orderBy(F.desc('ratio_ticket_por_venda')).select(
    'nome_produto', 'qtd_vendida', 'qtd_tickets_gerados', 'ratio_ticket_por_venda', 'tipo_problema_mais_frequente'
).show(5, truncate=False)

gold_desempenho_produto gravada: 517 produtos
Top 5 por ratio_ticket_por_venda:
+-------------------------+-----------+-------------------+----------------------+----------------------------+
|nome_produto             |qtd_vendida|qtd_tickets_gerados|ratio_ticket_por_venda|tipo_problema_mais_frequente|
+-------------------------+-----------+-------------------+----------------------+----------------------------+
|Carregador Turbo         |4593       |415                |0.0904                |Reembolso                   |
|Caixa de Som Bluetooth   |4688       |392                |0.0836                |Reembolso                   |
|Teclado sem Fio          |4606       |383                |0.0832                |Produto                     |
|Suporte para Monitor     |4738       |390                |0.0823                |Reembolso                   |
|Bateria Portátil 20000mAh|4599       |372                |0.0809                |Reembolso                   |
+-----------------------

---

## Tabelas: `gold_analise_suporte_por_tipo` e `gold_analise_suporte_por_agente`

**Origem:** `silver.ft_tickets_suporte`, `silver.dim_tipos_problema`, `silver.dim_agentes_suporte`  
**Destino:** `gold.gold_analise_suporte_por_tipo`, `gold.gold_analise_suporte_por_agente`  
**Stakeholder:** Time de Suporte / Gestão do SAC  
**Descrição:** Duas tabelas complementares que alimentam o dashboard de suporte. A primeira oferece visão por tipo de problema; a segunda, visão por agente.

### Schema — `gold_analise_suporte_por_tipo`

| Coluna | Tipo | Origem | Descrição |
|---|---|---|---|
| `tipo_problema` | string | ft_tickets_suporte | Tipo de problema canônico |
| `categoria_problema` | string | dim_tipos_problema | Categoria de negócio do problema |
| `total_tickets` | long | ft_tickets_suporte | Total de tickets deste tipo |
| `tickets_resolvidos` | long | ft_tickets_suporte | Tickets com resolução registrada |
| `taxa_resolucao` | decimal | derivado | `tickets_resolvidos / total_tickets` |
| `tempo_medio_resolucao_horas` | decimal | ft_tickets_suporte | Média de `tempo_resolucao_horas` (apenas resolvidos) |
| `nota_media_atendimento` | decimal | ft_tickets_suporte | Média de `nota_avaliacao` |
| `timestamp_ingestion` | timestamp | derivado | Instante de geração desta tabela Gold |

### Schema — `gold_analise_suporte_por_agente`

| Coluna | Tipo | Origem | Descrição |
|---|---|---|---|
| `agente_suporte` | string | ft_tickets_suporte | Nome do agente |
| `total_tickets` | long | ft_tickets_suporte | Total de tickets atendidos |
| `tickets_resolvidos` | long | ft_tickets_suporte | Tickets com resolução registrada |
| `taxa_resolucao` | decimal | derivado | `tickets_resolvidos / total_tickets` |
| `tempo_medio_resolucao_horas` | decimal | ft_tickets_suporte | Média de `tempo_resolucao_horas` |
| `nota_media_atendimento` | decimal | ft_tickets_suporte | Média de `nota_avaliacao` |
| `timestamp_ingestion` | timestamp | derivado | Instante de geração desta tabela Gold |

In [0]:
tickets      = spark.table(f'{silver_schema}.ft_tickets_suporte')
dim_prob     = spark.table(f'{silver_schema}.dim_tipos_problema')

suporte_tipo = (
    tickets
    .groupBy('tipo_problema')
    .agg(
        F.count('ticket_id').alias('total_tickets'),
        F.sum(F.when(F.col('resolvido') == True, 1).otherwise(0)).alias('tickets_resolvidos'),
        F.round(F.avg(
            F.when(F.col('resolvido') == True, F.col('tempo_resolucao_horas'))
        ), 2).alias('tempo_medio_resolucao_horas'),
        F.round(F.avg('nota_avaliacao'), 2).alias('nota_media_atendimento'),
    )
    .join(dim_prob, on='tipo_problema', how='left')
    .withColumn(
        'taxa_resolucao',
        F.round(F.col('tickets_resolvidos') / F.col('total_tickets'), 4)
    )
    .withColumn('timestamp_ingestion', F.current_timestamp())
    .select(
        'tipo_problema', 'categoria_problema', 'total_tickets',
        'tickets_resolvidos', 'taxa_resolucao',
        'tempo_medio_resolucao_horas', 'nota_media_atendimento',
        'timestamp_ingestion'
    )
    .orderBy(F.desc('total_tickets'))
)

suporte_tipo.write.format('delta').mode('overwrite').saveAsTable(f'{gold_schema}.gold_analise_suporte_por_tipo')

print(f'gold_analise_suporte_por_tipo gravada: {suporte_tipo.count()} tipos de problema')
suporte_tipo.show(truncate=False)

gold_analise_suporte_por_tipo gravada: 4 tipos de problema
+-------------+------------------+-------------+------------------+--------------+---------------------------+----------------------+--------------------------+
|tipo_problema|categoria_problema|total_tickets|tickets_resolvidos|taxa_resolucao|tempo_medio_resolucao_horas|nota_media_atendimento|timestamp_ingestion       |
+-------------+------------------+-------------+------------------+--------------+---------------------------+----------------------+--------------------------+
|Entrega      |Logística         |10517        |9860              |0.9375        |119.73                     |3.65                  |2026-05-06 06:33:18.650818|
|Reembolso    |Financeiro        |8903         |8383              |0.9416        |121.43                     |3.36                  |2026-05-06 06:33:18.650818|
|Produto      |Qualidade         |8574         |8065              |0.9406        |120.07                     |3.37                  |202

In [0]:
tickets = spark.table(f'{silver_schema}.ft_tickets_suporte')

suporte_agente = (
    tickets
    .groupBy('agente_suporte')
    .agg(
        F.count('ticket_id').alias('total_tickets'),
        F.sum(F.when(F.col('resolvido') == True, 1).otherwise(0)).alias('tickets_resolvidos'),
        F.round(F.avg(
            F.when(F.col('resolvido') == True, F.col('tempo_resolucao_horas'))
        ), 2).alias('tempo_medio_resolucao_horas'),
        F.round(F.avg('nota_avaliacao'), 2).alias('nota_media_atendimento'),
    )
    .withColumn(
        'taxa_resolucao',
        F.round(F.col('tickets_resolvidos') / F.col('total_tickets'), 4)
    )
    .withColumn('timestamp_ingestion', F.current_timestamp())
    .select(
        'agente_suporte', 'total_tickets', 'tickets_resolvidos',
        'taxa_resolucao', 'tempo_medio_resolucao_horas',
        'nota_media_atendimento', 'timestamp_ingestion'
    )
    .orderBy(F.desc('total_tickets'))
)

suporte_agente.write.format('delta').mode('overwrite').saveAsTable(f'{gold_schema}.gold_analise_suporte_por_agente')

print(f'gold_analise_suporte_por_agente gravada: {suporte_agente.count()} agentes')
suporte_agente.show(10, truncate=False)

gold_analise_suporte_por_agente gravada: 20 agentes
+--------------------+-------------+------------------+--------------+---------------------------+----------------------+--------------------------+
|agente_suporte      |total_tickets|tickets_resolvidos|taxa_resolucao|tempo_medio_resolucao_horas|nota_media_atendimento|timestamp_ingestion       |
+--------------------+-------------+------------------+--------------+---------------------------+----------------------+--------------------------+
|Carlos Eduardo Silva|6143         |5793              |0.943         |118.51                     |3.76                  |2026-05-06 06:33:23.745178|
|Ana Paula Ferreira  |5924         |5589              |0.9435        |118.77                     |3.80                  |2026-05-06 06:33:23.745178|
|Mariana Costa       |5209         |4924              |0.9453        |121.24                     |3.71                  |2026-05-06 06:33:23.745178|
|Fernanda Lima       |2461         |2325              

---

## Tabela: `gold_satisfacao_nps`

**Origem:** `silver.ft_avaliacoes`, `silver.ft_pedidos`, `silver.dim_produtos`  
**Destino:** `gold.gold_satisfacao_nps`  
**Stakeholder:** Diretoria / Qualquer área com visão executiva  
**Descrição:** Uma linha por `ano_mes x categoria` com métricas de NPS e satisfação consolidadas. O `nps_score` é calculado como `% Promotores − % Detratores`, na escala de −100 a +100.

### Classificação NPS (regra aplicada na Silver)

| Categoria | Nota NPS |
|---|---|
| `Promotor` | 9 ou 10 |
| `Neutro` | 7 ou 8 |
| `Detrator` | 0 a 6 |

### Schema

| Coluna | Tipo | Origem | Descrição |
|---|---|---|---|
| `ano_mes` | string | ft_pedidos | Período no formato `YYYY-MM` |
| `categoria` | string | dim_produtos | Categoria do produto avaliado |
| `total_avaliacoes` | long | ft_avaliacoes | Total de avaliações no período e categoria |
| `nota_produto_media` | decimal | ft_avaliacoes | Média das notas do produto (0–5) |
| `nota_nps_media` | decimal | ft_avaliacoes | Média das notas NPS (0–10) |
| `qtd_promotores` | long | ft_avaliacoes | Avaliações com `categoria_nps = 'Promotor'` |
| `qtd_neutros` | long | ft_avaliacoes | Avaliações com `categoria_nps = 'Neutro'` |
| `qtd_detratores` | long | ft_avaliacoes | Avaliações com `categoria_nps = 'Detrator'` |
| `pct_promotores` | decimal | derivado | `qtd_promotores / total_avaliacoes x 100` |
| `pct_neutros` | decimal | derivado | `qtd_neutros / total_avaliacoes x 100` |
| `pct_detratores` | decimal | derivado | `qtd_detratores / total_avaliacoes x 100` |
| `nps_score` | decimal | derivado | `pct_promotores − pct_detratores` (−100 a +100) |
| `pct_recomenda` | decimal | derivado | `% de avaliações com recomenda = true` |
| `timestamp_ingestion` | timestamp | derivado | Instante de geração desta tabela Gold |

In [0]:
avaliacoes = spark.table(f'{silver_schema}.ft_avaliacoes')
pedidos    = spark.table(f'{silver_schema}.ft_pedidos').select('id_pedido', 'ano_mes')
produtos   = spark.table(f'{silver_schema}.dim_produtos').select('id_produto', 'categoria')

# ─── Enriquecer avaliações com ano_mes (via pedido) e categoria (via produto) ─
aval_enriquecida = (
    avaliacoes
    .join(pedidos,  on='id_pedido',  how='left')
    .join(produtos, on='id_produto', how='left')
)

# ─── Agregação por ano_mes x categoria ────────────────────────────────────────
nps = (
    aval_enriquecida
    .groupBy('ano_mes', 'categoria')
    .agg(
        F.count('id_avaliacao').alias('total_avaliacoes'),
        F.round(F.avg('nota_produto'), 2).alias('nota_produto_media'),
        F.round(F.avg('nota_nps'), 2).alias('nota_nps_media'),
        F.sum(F.when(F.col('categoria_nps') == 'Promotor', 1).otherwise(0)).alias('qtd_promotores'),
        F.sum(F.when(F.col('categoria_nps') == 'Neutro',   1).otherwise(0)).alias('qtd_neutros'),
        F.sum(F.when(F.col('categoria_nps') == 'Detrator', 1).otherwise(0)).alias('qtd_detratores'),
        F.sum(F.when(F.col('recomenda') == True, 1).otherwise(0)).alias('qtd_recomenda'),
    )
    # ─── Percentuais e NPS Score ──────────────────────────────────────────────
    .withColumn('pct_promotores', F.round(F.col('qtd_promotores') / F.col('total_avaliacoes') * 100, 2))
    .withColumn('pct_neutros',    F.round(F.col('qtd_neutros')    / F.col('total_avaliacoes') * 100, 2))
    .withColumn('pct_detratores', F.round(F.col('qtd_detratores') / F.col('total_avaliacoes') * 100, 2))
    .withColumn('nps_score',      F.round(F.col('pct_promotores') - F.col('pct_detratores'), 2))
    .withColumn('pct_recomenda',  F.round(F.col('qtd_recomenda')  / F.col('total_avaliacoes') * 100, 2))
    .withColumn('timestamp_ingestion', F.current_timestamp())
    .drop('qtd_recomenda')
    .orderBy('ano_mes', 'categoria')
)

nps.write.format('delta').mode('overwrite').saveAsTable(f'{gold_schema}.gold_satisfacao_nps')

print(f'gold_satisfacao_nps gravada: {nps.count():,} combinações ano_mes x categoria')
print('NPS Score por categoria (média geral):')
nps.groupBy('categoria').agg(
    F.round(F.avg('nps_score'), 1).alias('nps_medio'),
    F.sum('total_avaliacoes').alias('total_avaliacoes')
).orderBy(F.desc('nps_medio')).show(truncate=False)

gold_satisfacao_nps gravada: 369 combinações ano_mes x categoria
NPS Score por categoria (média geral):
+-----------+---------+----------------+
|categoria  |nps_medio|total_avaliacoes|
+-----------+---------+----------------+
|Vestuario  |16.9     |11424           |
|Esportes   |16.7     |11008           |
|Moveis     |16.4     |6620            |
|Beleza     |16.3     |15106           |
|Casa       |15.8     |12720           |
|Automotivo |15.6     |10958           |
|Indefinida |15.2     |10592           |
|Brinquedos |14.9     |12618           |
|Eletronicos|11.2     |65786           |
+-----------+---------+----------------+



---

## Tabela: `gold_pedidos_detalhado`

**Origem:** `silver.ft_pedidos`, `silver.dim_produtos`  
**Destino:** `gold.gold_pedidos_detalhado`  
**Stakeholder:** Diretoria Comercial / Operações / CRM / Agente de IA  
**Descrição:** Tabela Gold detalhada de pedidos, com uma linha por pedido/produto. Consolida informações tratadas da Silver, enriquece os pedidos com dados do produto e disponibiliza valores financeiros padronizados para consumo pelo CRM, dashboards e agente de IA.

### Tratamento de quantidade nula

Pedidos com `quantidade` nula não têm a quantidade inferida a partir do valor do pedido, pois os próprios valores de origem apresentam inconsistências. Nesses casos, o pedido permanece disponível para análises operacionais e financeiras quando houver valor tratado disponível, mas não contribui corretamente para métricas baseadas em quantidade de itens.

### Schema

| Coluna | Tipo | Origem | Descrição |
|---|---|---|---|
| `id_pedido` | string | ft_pedidos | Identificador único do pedido |
| `id_cliente` | string | ft_pedidos | Identificador do cliente associado ao pedido |
| `id_produto` | string | ft_pedidos | Identificador do produto associado ao pedido |
| `nome_produto` | string | dim_produtos | Nome do produto associado ao pedido |
| `categoria` | string | dim_produtos | Categoria normalizada do produto |
| `ativo` | boolean | dim_produtos | Indica se o produto está ativo no catálogo |
| `data_pedido` | date | ft_pedidos | Data padronizada do pedido |
| `ano_mes` | string | ft_pedidos | Período do pedido no formato `YYYY-MM` |
| `metodo_pagamento` | string | ft_pedidos | Método de pagamento padronizado |
| `status` | string | ft_pedidos | Status padronizado do pedido |
| `quantidade` | int | ft_pedidos | Quantidade de itens no pedido |
| `preco_produto` | decimal | dim_produtos | Preço tratado do produto |
| `valor_pedido_final` | decimal | derivado | Valor financeiro final utilizado nas análises de negócio |
| `receita_bruta` | decimal | derivado | Valor considerado como receita para pedidos aprovados |
| `valor_reembolsado` | decimal | derivado | Valor associado a pedidos reembolsados |
| `timestamp_ingestion` | timestamp | derivado | Instante de geração desta tabela Gold |

In [0]:
# ─── Leitura das tabelas de origem ────────────────────────────────────────────
# A tabela ft_pedidos já passou pelos tratamentos da Silver:
# - padronização de status
# - padronização de método de pagamento
# - tratamento de datas
# - tratamento de valores e quantidades
#
# A dimensão de produtos é usada para enriquecer os pedidos com informações
# de catálogo, principalmente preço e categoria.

pedidos = spark.table(f'{silver_schema}.ft_pedidos')

produtos = spark.table(f'{silver_schema}.dim_produtos').select(
    'id_produto',
    'nome_produto',
    'categoria',
    F.col('preco').alias('preco_produto'),
    'ativo'
)

print(f'Pedidos Silver: {pedidos.count():,}')
print(f'Produtos Silver: {produtos.count():,}')

Pedidos Silver: 314,900
Produtos Silver: 517


In [0]:
# ─── Enriquecimento dos pedidos com dados do produto ──────────────────────────
# O join adiciona ao pedido informações descritivas do produto vendido.
# O valor financeiro já foi corrigido na camada Silver, portanto a Gold
# não recalcula mais valor_pedido com base em quantidade x preço do produto.

produtos_info = (
    produtos
    .select(
        'id_produto',
        'nome_produto',
        'categoria',
        'ativo'
    )
)

pedidos_enriquecidos = (
    pedidos
    .join(produtos_info, on='id_produto', how='left')
)

# ─── Construção da tabela Gold detalhada de pedidos ───────────────────────────
# Esta tabela será usada como base para os demais Data Marts de pedidos.
#
# Decisões aplicadas:
#
# 1. O valor financeiro principal da Gold é valor_pedido.
#    Ele utiliza diretamente o valor_pedido corrigido na Silver.
#
# 2. A correção financeira por quantidade x preço do produto não é refeita aqui,
#    pois essa regra foi consolidada na silver.ft_pedidos.
#
# 3. As flags técnicas de qualidade não são expostas na Gold final.
#    A Gold deve ser uma camada de consumo de negócio para CRM, dashboards
#    e agente de IA.

gold_pedidos_detalhado = (
    pedidos_enriquecidos

    # Receita bruta considera apenas pedidos aprovados.
    # Pedidos recusados ou em processamento não devem compor receita realizada.
    .withColumn(
        'receita_bruta',
        F.when(
            F.col('status') == 'Aprovado',
            F.col('valor_pedido')
        ).otherwise(F.lit(0))
    )

    # Valor reembolsado é separado para permitir análise de perdas/devoluções.
    .withColumn(
        'valor_reembolsado',
        F.when(
            F.col('status') == 'Reembolsado',
            F.col('valor_pedido')
        ).otherwise(F.lit(0))
    )

    # Marca o momento de geração da Gold para rastreabilidade do pipeline.
    .withColumn('timestamp_ingestion', F.current_timestamp())

    # Seleciona apenas colunas úteis para consumo analítico.
    .select(
        'id_pedido',
        'id_cliente',
        'id_produto',
        'nome_produto',
        'categoria',
        'ativo',
        'data_pedido',
        'ano_mes',
        'metodo_pagamento',
        'status',
        'quantidade',
        F.round('valor_pedido', 2).alias('valor_pedido'),
        F.round('receita_bruta', 2).alias('receita_bruta'),
        F.round('valor_reembolsado', 2).alias('valor_reembolsado'),
        'timestamp_ingestion'
    )

    # Ordenação apenas para facilitar leitura e validação no notebook.
    .orderBy('data_pedido', 'id_pedido')
)

gold_pedidos_detalhado.write \
    .format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable(f'{gold_schema}.gold_pedidos_detalhado')

print(f'gold_pedidos_detalhado gravada: {gold_pedidos_detalhado.count():,} registros')

gold_pedidos_detalhado gravada: 314,900 registros


In [0]:
display(gold_pedidos_detalhado)

id_pedido,id_cliente,id_produto,nome_produto,categoria,ativo,data_pedido,ano_mes,metodo_pagamento,status,quantidade,valor_pedido,receita_bruta,valor_reembolsado,timestamp_ingestion
00388693-f07f-45ee-80b5-607c0ceca13d,da2a251e-e2dd-4885-b8a9-58be529035f2,PROD-0024,Mouse Sem Fio,Eletronicos,false,2023-01-01,2023-01,Boleto,Aprovado,null,221.36,221.36,0.00,2026-05-06T06:33:39.306Z
014935a8-1448-4a4f-8ff5-c5c101a87d75,f6947772-ce86-45cf-8b67-7ad4bafecbe5,PROD-0386,Pelucia Gigante 80cm,Brinquedos,true,2023-01-01,2023-01,Pix,Aprovado,4,660.00,660.00,0.00,2026-05-06T06:33:39.306Z
01a4070a-93d7-4ff7-b4bc-5c6e0c7e66ac,98e6b47c-1f48-4c18-9605-8b21a63aaf52,PROD-0016,Caixa de Som Bluetooth,Eletronicos,true,2023-01-01,2023-01,Pix,Aprovado,5,1604.31,1604.31,0.00,2026-05-06T06:33:39.306Z
03ff41ad-833d-48c3-9feb-862c43a6c978,969f4bb8-3252-49a0-a163-596f84cd295d,PROD-0016,Caixa de Som Bluetooth,Eletronicos,true,2023-01-01,2023-01,Cartão,Aprovado,4,1310.51,1310.51,0.00,2026-05-06T06:33:39.306Z
05674a48-54be-4dbd-957a-3e7b94cfcfd9,39ad15c7-42fb-4bc3-8e54-533bf5a79ec0,PROD-0430,Gym para Bebe,Brinquedos,true,2023-01-01,2023-01,Pix,Aprovado,3,717.00,717.00,0.00,2026-05-06T06:33:39.306Z
065d1642-7aca-4e09-a6de-2441105946a0,491870aa-0c65-4bf9-ae25-6b23ebcc74ff,PROD-0056,Nobreak 1000VA,Eletronicos,false,2023-01-01,2023-01,Cartão,Aprovado,1,518.00,518.00,0.00,2026-05-06T06:33:39.306Z
0a1eaa43-7ba4-4b1b-a772-ba1e0b340d00,2447b3cd-a152-42dd-a925-2c497a0eb55b,PROD-0073,Moletom com Capuz,Vestuario,true,2023-01-01,2023-01,Cartão,Aprovado,3,194.44,194.44,0.00,2026-05-06T06:33:39.306Z
0f31600a-444c-4a42-8743-b0fdb03da8a2,91473045-4ca5-407b-bb97-f0b17465d4a6,PROD-0226,Esteira Ergométrica,Esportes,true,2023-01-01,2023-01,Pix,Aprovado,4,240.35,240.35,0.00,2026-05-06T06:33:39.306Z
0fd8fe2b-98d6-4d92-a4b3-9a73b2085c62,01ce9a2a-1600-47cd-a61c-4b6baa3e2b33,PROD-0047,Cabo USB-C 2m,Indefinida,null,2023-01-01,2023-01,Cartão,Aprovado,2,118.00,118.00,0.00,2026-05-06T06:33:39.306Z
0ff798da-9884-4546-937e-75bd29a3229e,2a4906c6-2a0e-421f-b965-1952d445268a,PROD-0372,Boneca Articulada 30cm,Brinquedos,true,2023-01-01,2023-01,Pix,Aprovado,2,290.00,290.00,0.00,2026-05-06T06:33:39.306Z


---

## Tabela: `gold_pedidos_por_status`

**Origem:** `gold.gold_pedidos_detalhado`  
**Destino:** `gold.gold_pedidos_por_status`  
**Stakeholder:** Operações / Diretoria Comercial  
**Descrição:** Uma linha por `ano_mes x status`, consolidando o volume de pedidos e a participação percentual de cada status no mês. Esta tabela permite acompanhar a distribuição operacional dos pedidos e identificar variações em recusas, reembolsos, aprovações e pedidos em processamento.

### Regras de negócio

| Métrica | Regra |
|---|---|
| `qtd_pedidos` | Quantidade distinta de pedidos por mês e status |
| `percentual_pedidos` | Percentual de pedidos do status em relação ao total de pedidos do mês |

### Observação sobre métricas financeiras

Métricas financeiras, como receita e ticket médio, não são incluídas aqui, pois esta tabela tem foco operacional. Essas métricas devem ser consultadas em tabelas analíticas de vendas, como `gold_vendas_mensais`, que utiliza os valores financeiros já corrigidos na Silver.

### Schema

| Coluna | Tipo | Origem | Descrição |
|---|---|---|---|
| `ano_mes` | string | gold_pedidos_detalhado | Período no formato `YYYY-MM` |
| `status` | string | gold_pedidos_detalhado | Status padronizado do pedido |
| `qtd_pedidos` | long | derivado | Quantidade distinta de pedidos no mês e status |
| `percentual_pedidos` | decimal | derivado | Participação percentual do status dentro do total mensal de pedidos |
| `timestamp_ingestion` | timestamp | derivado | Instante de geração desta tabela Gold |

In [0]:
# ─── Leitura da tabela Gold detalhada de pedidos ──────────────────────────────
# A gold_pedidos_detalhado contém a base completa de pedidos tratados.
#
# Como esta tabela tem foco operacional, usamos todos os pedidos disponíveis,
# inclusive aqueles sem base segura para cálculo financeiro.

pedidos_detalhado = spark.table(f'{gold_schema}.gold_pedidos_detalhado')

print(f'Pedidos detalhados Gold: {pedidos_detalhado.count():,}')

Pedidos detalhados Gold: 314,900


In [0]:
# ─── Construção da tabela gold_pedidos_por_status ─────────────────────────────
# Objetivo:
# consolidar a distribuição mensal de pedidos por status.
#
# Regras aplicadas:
# - qtd_pedidos: quantidade distinta de pedidos por ano_mes e status
# - percentual_pedidos: participação do status dentro do total de pedidos do mês
#
# Esta tabela não inclui valores financeiros, pois seu foco é operacional.

from pyspark.sql.window import Window

# Janela mensal usada para calcular o percentual de participação de cada status
# dentro do total de pedidos do respectivo mês.
w_mes = Window.partitionBy('ano_mes')

gold_pedidos_por_status = (
    pedidos_detalhado
    .groupBy(
        'ano_mes',
        'status'
    )
    .agg(
        F.countDistinct('id_pedido').alias('qtd_pedidos')
    )

    # Calcula a participação percentual do status no mês.
    # Exemplo: se em um mês houve 100 pedidos e 70 foram aprovados,
    # o percentual de aprovados será 70%.
    .withColumn(
        'percentual_pedidos',
        F.round(
            (F.col('qtd_pedidos') / F.sum('qtd_pedidos').over(w_mes)) * 100,
            2
        )
    )

    # Timestamp de geração para rastreabilidade do pipeline.
    .withColumn('timestamp_ingestion', F.current_timestamp())

    # Organização das colunas na ordem documentada.
    .select(
        'ano_mes',
        'status',
        'qtd_pedidos',
        'percentual_pedidos',
        'timestamp_ingestion'
    )

    # Ordenação apenas para facilitar leitura no notebook.
    .orderBy('ano_mes', 'status')
)

# ─── Escrita da tabela Gold ───────────────────────────────────────────────────
# overwriteSchema garante que mudanças no schema sejam refletidas na tabela Delta.

gold_pedidos_por_status.write \
    .format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable(f'{gold_schema}.gold_pedidos_por_status')

print(f'gold_pedidos_por_status gravada: {gold_pedidos_por_status.count():,} registros')

gold_pedidos_por_status gravada: 164 registros


---

## Tabela: `gold_vendas_mensais`

**Origem:** `gold.gold_pedidos_detalhado` 
**Destino:** `gold.gold_vendas_mensais`  
**Stakeholder:** Diretoria Comercial / Gestão Executiva / Agente de IA  
**Descrição:** Uma linha por `ano_mes x metodo_pagamento`, consolidando métricas mensais de vendas, receita, reembolsos, ticket médio, clientes e produtos. Esta tabela alimenta dashboards comerciais e consultas do agente de IA sobre desempenho mensal de vendas.

### Regras de negócio

| Métrica | Regra |
|---|---|
| `qtd_pedidos` | Quantidade distinta de pedidos no mês e método de pagamento |
| `qtd_pedidos_aprovados` | Quantidade distinta de pedidos aprovados |
| `qtd_pedidos_reembolsados` | Quantidade distinta de pedidos reembolsados |
| `qtd_clientes` | Quantidade distinta de clientes no agrupamento |
| `qtd_produtos` | Quantidade distinta de produtos no agrupamento |
| `qtd_itens_vendidos` | Soma da quantidade dos pedidos aprovados |
| `receita_bruta` | Soma da `receita_bruta` dos pedidos aprovados |
| `valor_reembolsado` | Soma do `valor_reembolsado` dos pedidos reembolsados |
| `ticket_medio` | `receita_bruta / qtd_pedidos_aprovados` |

### Observação sobre base financeira

Esta tabela utiliza a `gold_pedidos_detalhado`, que já recebe o `valor_pedido` corrigido a partir da Silver. Dessa forma, métricas como receita, reembolso e ticket médio são calculadas a partir do valor financeiro oficial consolidado anteriormente na camada Silver.

### Schema

| Coluna | Tipo | Origem | Descrição |
|---|---|---|---|
| `ano_mes` | string | gold_pedidos_detalhado | Período no formato `YYYY-MM` |
| `metodo_pagamento` | string | gold_pedidos_detalhado | Método de pagamento padronizado |
| `qtd_pedidos` | long | derivado | Quantidade distinta de pedidos no agrupamento |
| `qtd_pedidos_aprovados` | long | derivado | Quantidade distinta de pedidos aprovados |
| `qtd_pedidos_reembolsados` | long | derivado | Quantidade distinta de pedidos reembolsados |
| `qtd_clientes` | long | derivado | Quantidade distinta de clientes no agrupamento |
| `qtd_produtos` | long | derivado | Quantidade distinta de produtos no agrupamento |
| `qtd_itens_vendidos` | long | derivado | Soma da quantidade de itens de pedidos aprovados |
| `receita_bruta` | decimal | derivado | Receita total de pedidos aprovados |
| `valor_reembolsado` | decimal | derivado | Valor total associado a pedidos reembolsados |
| `ticket_medio` | decimal | derivado | Receita bruta dividida pela quantidade de pedidos aprovados |
| `timestamp_ingestion` | timestamp | derivado | Instante de geração desta tabela Gold |

In [0]:
# ─── Leitura da tabela Gold detalhada de pedidos ──────────────────────────────
# A gold_pedidos_detalhado contém os pedidos enriquecidos com informações de produto
#
# Esta será a base para a tabela mensal de vendas.

pedidos_vendas = spark.table(f'{gold_schema}.gold_pedidos_detalhado')

print(f'Pedidos detalhados Gold para vendas mensais: {pedidos_vendas.count():,}')

Pedidos detalhados Gold para vendas mensais: 314,900


In [0]:
# ─── Construção da tabela gold_vendas_mensais ─────────────────────────────────
# Objetivo:
# consolidar as principais métricas comerciais por mês e método de pagamento.
#
# Esta tabela é voltada para dashboard comercial e perguntas executivas, como:
# - qual foi a receita por mês?
# - qual foi o ticket médio mensal?
# - qual método de pagamento concentrou mais vendas?
# - quanto foi reembolsado por período?

gold_vendas_mensais = (
    pedidos_vendas
    .groupBy(
        'ano_mes',
        'metodo_pagamento'
    )
    .agg(
        # Volume geral de pedidos no agrupamento.
        F.countDistinct('id_pedido').alias('qtd_pedidos'),

        # Volume de pedidos aprovados, usado também no cálculo do ticket médio.
        F.countDistinct(
            F.when(F.col('status') == 'Aprovado', F.col('id_pedido'))
        ).alias('qtd_pedidos_aprovados'),

        # Volume de pedidos reembolsados.
        F.countDistinct(
            F.when(F.col('status') == 'Reembolsado', F.col('id_pedido'))
        ).alias('qtd_pedidos_reembolsados'),

        # Clientes e produtos distintos no mês/método de pagamento.
        F.countDistinct('id_cliente').alias('qtd_clientes'),
        F.countDistinct('id_produto').alias('qtd_produtos'),

        # Quantidade vendida considera apenas pedidos aprovados.
        F.sum(
            F.when(F.col('status') == 'Aprovado', F.col('quantidade')).otherwise(0)
        ).alias('qtd_itens_vendidos'),

        # Receita bruta já foi calculada na base financeira apenas para aprovados.
        F.round(F.sum('receita_bruta'), 2).alias('receita_bruta'),

        # Valor reembolsado já foi calculado na base financeira apenas para reembolsados.
        F.round(F.sum('valor_reembolsado'), 2).alias('valor_reembolsado')
    )

    # Ticket médio considera apenas pedidos aprovados.
    # Quando não houver pedido aprovado no agrupamento, o ticket fica nulo.
    .withColumn(
        'ticket_medio',
        F.when(
            F.col('qtd_pedidos_aprovados') > 0,
            F.round(F.col('receita_bruta') / F.col('qtd_pedidos_aprovados'), 2)
        )
    )

    # Timestamp de geração para rastreabilidade do pipeline.
    .withColumn('timestamp_ingestion', F.current_timestamp())

    # Organização das colunas na ordem documentada.
    .select(
        'ano_mes',
        'metodo_pagamento',
        'qtd_pedidos',
        'qtd_pedidos_aprovados',
        'qtd_pedidos_reembolsados',
        'qtd_clientes',
        'qtd_produtos',
        'qtd_itens_vendidos',
        'receita_bruta',
        'valor_reembolsado',
        'ticket_medio',
        'timestamp_ingestion'
    )

    # Ordenação apenas para facilitar leitura no notebook.
    .orderBy('ano_mes', 'metodo_pagamento')
)

# ─── Escrita da tabela Gold ───────────────────────────────────────────────────
# overwriteSchema garante que mudanças no schema sejam refletidas na tabela Delta.

gold_vendas_mensais.write \
    .format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable(f'{gold_schema}.gold_vendas_mensais')

print(f'gold_vendas_mensais gravada: {gold_vendas_mensais.count():,} registros')

gold_vendas_mensais gravada: 123 registros


---

## Exportação para CSV

As tabelas Gold são exportadas em formato CSV para popular o banco SQLite local que serve o backend FastAPI do CRM.  
Ajuste o caminho `export_path` conforme o destino configurado no ambiente Databricks (DBFS ou Volume).

In [0]:
import os

gold_tables = [
    'gold_cliente_360',
    'gold_kpis_vendas_mensal',
    'gold_vendas_por_dimensao',
    'gold_desempenho_produto',
    'gold_analise_suporte_por_tipo',
    'gold_analise_suporte_por_agente',
    'gold_satisfacao_nps',
    'gold_pedidos_detalhado',
    'gold_pedidos_por_status',
    'gold_vendas_mensais',
]

gold_base_path = '/Workspace/Users/chab@cin.ufpe.br/v-commerce-crm-360/gold_csv'
os.makedirs(gold_base_path, exist_ok=True)

for table in gold_tables:
    df_spark = spark.table(f'{gold_schema}.{table}')
    df_pandas = df_spark.toPandas()
    output_path = os.path.join(gold_base_path, f'{table}.csv')
    df_pandas.to_csv(output_path, index=False)
    print(f'Exportado: {table} ({len(df_pandas):,} linhas)')

print(f'\nExportação concluída. Arquivos em: {gold_base_path}/')

Exportado: gold_cliente_360 (58,322 linhas)
Exportado: gold_kpis_vendas_mensal (41 linhas)
Exportado: gold_vendas_por_dimensao (2,213 linhas)
Exportado: gold_desempenho_produto (517 linhas)
Exportado: gold_analise_suporte_por_tipo (4 linhas)
Exportado: gold_analise_suporte_por_agente (20 linhas)
Exportado: gold_satisfacao_nps (369 linhas)
Exportado: gold_pedidos_detalhado (314,900 linhas)
Exportado: gold_pedidos_por_status (164 linhas)
Exportado: gold_vendas_mensais (123 linhas)

Exportação concluída. Arquivos em: /Workspace/Users/gabrielbzandrade@gmail.com/v-commerce-crm-360/gold_csv/
